<a href="https://colab.research.google.com/github/PedroCorreia3880/PedroCorreia3880.github.io/blob/main/IAC_TP1_A_spambase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IAC · Trabalho 1 · Conjunto A · Spambase

**Inteligência Artificial em Cibersegurança** · CTeSP Cibersegurança · ISLA Gaia · 2026/2027 · Mário Amorim

**Grupo:** _escrevam aqui o número do grupo_ · **Elementos:** _nome 1, nome 2, nome 3_

---

**O problema:** spam em email. **Os dados:** Cada linha é um email de 1999, já transformado em 57 números: a frequência de 48 palavras e 6 caracteres (por exemplo `word_freq_free`, `char_freq_$`) e três medidas das sequências de maiúsculas. Não há o texto original.

**Fonte (citem-na no relatório):** Hopkins, M., Reeber, E., Forman, G. e Suermondt, J. (1999). *Spambase*. UCI Machine Learning Repository, conjunto 94.

**A vossa pergunta obrigatória** (é feita na defesa): As características foram escolhidas em 1999 e algumas vêm da caixa de correio de uma única pessoa. Este detector serve para 2026? Como é que o testariam?

**Datas:** oficina a 6 de Novembro (tragam a Parte 1 feita, o cenário de custo e um primeiro modelo) · entrega no Moodle até
quinta-feira, 12 de Novembro, às 23h59 · defesa a 13 de Novembro.

**Antes de começar:** Ficheiro → Guardar uma cópia no Drive. Este caderno é o esqueleto: as células de código correm de
ponta a ponta com valores por omissão, e o trabalho é vosso nos sítios marcados com **A FAZER**.

## Parte 0 · Carregar os dados

In [ ]:
import os, time, warnings
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
warnings.filterwarnings('ignore')

FICHEIRO = 'A-spambase.csv'
if not os.path.exists(FICHEIRO):
    from google.colab import files
    files.upload()                      # escolham o ficheiro A-spambase.csv do Moodle

d = pd.read_csv(FICHEIRO)
print(d.shape[0], 'linhas ·', d.shape[1], 'colunas')
d.head()

## Parte 1 · Explorar antes de treinar (passo 1 do enunciado)

In [ ]:
print('Classes (1 = positivo):', d.etiqueta.value_counts().to_dict(), '·', round(100 * d.etiqueta.mean(), 1), '% de positivos')
print('Valores em falta por coluna (só as que têm):', {k: int(v) for k, v in d.isna().sum().items() if v})
print('Linhas exactamente repetidas:', int(d.duplicated().sum()))

In [ ]:
IGNORAR = []            # colunas de texto que não entram nos modelos
cols = [x for x in d.columns if x not in IGNORAR + ['etiqueta']]
repetidos_x = int(d[cols].duplicated().sum())
contraditorios = int(d.groupby(cols).etiqueta.nunique().gt(1).sum()) if repetidos_x else 0
print('Linhas com as mesmas características (ignorando a etiqueta):', repetidos_x, '· das quais grupos com etiquetas diferentes:', contraditorios)

# a média de cada característica nas duas classes, ordenadas pela diferença relativa
med = d.groupby('etiqueta')[cols].mean().T
med['diferença relativa'] = (med[1] - med[0]).abs() / (d[cols].std() + 1e-9)
med.sort_values('diferença relativa', ascending=False).head(10).round(3)

In [ ]:
# A FAZER: escolham uma característica que vos pareça importante e vejam a distribuição nas duas classes
COLUNA = med.sort_values('diferença relativa', ascending=False).index[0]
for classe, cor in [(0, '#BB9184'), (1, '#762023')]:
    plt.hist(d.loc[d.etiqueta == classe, COLUNA], bins=30, alpha=.7, color=cor, label=f'etiqueta {classe}')
plt.xlabel(COLUNA); plt.ylabel('número de linhas'); plt.legend(); plt.show()

> **A FAZER (relatório, secção Dados):** quantas linhas, que colunas, que percentagem de positivos, valores em falta,
> repetidos, e o que viram no gráfico. **Decidam o que fazer com as linhas repetidas antes de dividir**, e expliquem porquê
> (PL02, Parte B). Se decidirem retirá-las, descomentem a linha da célula seguinte.

## Parte 2 · Dividir sem fugas (passo 2)

In [ ]:
# d = d.drop_duplicates(subset=cols).reset_index(drop=True)      # A FAZER: decidir (ver Parte 1)
X, y = d[cols], d.etiqueta

# a divisão de sempre: 70% treino, 30% teste, estratificada, semente 42. O teste só se usa uma vez, no fim.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
# do treino separa-se um quarto para validação: é aí que se escolhem os botões
X_sub, X_val, y_sub, y_val = train_test_split(X_tr, y_tr, test_size=0.25, random_state=0, stratify=y_tr)
print('treino', len(X_tr), '(sub-treino', len(X_sub), '+ validação', len(X_val), ') · teste', len(X_te))

def mede(nome, y_real, y_prev):
    vn, fp, fn, vp = confusion_matrix(y_real, y_prev).ravel()
    return {'modelo': nome, 'precisão': round(precision_score(y_real, y_prev, zero_division=0), 3),
            'revocação': round(recall_score(y_real, y_prev), 3), 'F1': round(f1_score(y_real, y_prev), 3), 'FP': int(fp), 'FN': int(fn)}

## Parte 3 · Três modelos, afinados (passo 3)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

# cada modelo recebe o valor do seu botão; os modelos lineares precisam das características na mesma escala
FAZ = {
    'Perceptrão':          lambda b: make_pipeline(StandardScaler(), Perceptron(random_state=0)),
    'SVM linear':          lambda b: make_pipeline(StandardScaler(), LinearSVC(C=b, max_iter=10000)),
    'Regressão logística': lambda b: make_pipeline(StandardScaler(), LogisticRegression(C=b, max_iter=5000)),
    'Árvore':              lambda b: DecisionTreeClassifier(max_depth=b, random_state=0),
    'Naive Bayes':         lambda b: GaussianNB(),        # para números, o Naive Bayes gaussiano: atenção à premissa
}
GRELHA = {'Perceptrão': [None], 'SVM linear': [0.01, 0.1, 1, 10], 'Regressão logística': [0.01, 0.1, 1, 10, 100],
          'Árvore': [2, 3, 5, 8, 12, None], 'Naive Bayes': [None]}

In [ ]:
def afina(nome):
    """Mede cada valor do botão na validação e devolve o melhor."""
    melhor, f1_melhor = None, -1
    for b in GRELHA[nome]:
        m = FAZ[nome](b).fit(X_sub, y_sub)
        f = f1_score(y_val, m.predict(X_val))
        print(f'{nome:<20} botão = {str(b):<6} F1 na validação = {f:.3f}')
        if f > f1_melhor:
            melhor, f1_melhor = b, f
    return melhor

# A FAZER: escolham pelo menos TRÊS dos cinco e justifiquem a escolha no relatório
ESCOLHIDOS = ['Regressão logística', 'Árvore', 'Naive Bayes']
BOTAO = {nome: afina(nome) for nome in ESCOLHIDOS}
BOTAO

In [ ]:
# o confronto: validação cruzada com 5 partes, só no treino (média e desvio) e o teste uma única vez
cinco = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
linhas, prev_cv = [], {}
for nome in ESCOLHIDOS:
    f1s, prev = [], np.zeros(len(X_tr), dtype=int)
    for a, b in cinco.split(X_tr, y_tr):
        m = FAZ[nome](BOTAO[nome]).fit(X_tr.iloc[a], y_tr.iloc[a])
        prev[b] = m.predict(X_tr.iloc[b]); f1s.append(f1_score(y_tr.iloc[b], prev[b]))
    prev_cv[nome] = prev
    t0 = time.perf_counter(); m = FAZ[nome](BOTAO[nome]).fit(X_tr, y_tr); ms = 1000 * (time.perf_counter() - t0)
    t = mede(nome, y_te, m.predict(X_te))
    linhas.append({'modelo': nome, 'botão': BOTAO[nome], 'F1 (5 partes)': f'{np.mean(f1s):.3f} ± {np.std(f1s):.3f}',
                   'F1 teste': t['F1'], 'FP teste': t['FP'], 'FN teste': t['FN'], 'treino (ms)': round(ms)})
pd.DataFrame(linhas)

> **A FAZER (relatório, secção Confronto):** copiem a tabela. Que modelos empatam (a diferença é menor do que o desvio)?
> Qual fica atrás, e porquê? Se algum resultado for **bom demais**, desconfiem antes de festejar (PL04).

## Parte 4 · Escolher pelo custo, e o mundo real (passo 4)

In [ ]:
# A FAZER: o vosso cenário. Quem usa o detector? Quanto custa um falso positivo e um falso negativo? Escrevam-no no relatório.
CENARIO = 'escrevam aqui o cenário'
CUSTO_FP, CUSTO_FN = 1, 1

custos = []
for nome, prev in prev_cv.items():
    vn, fp, fn, vp = confusion_matrix(y_tr, prev).ravel()
    custos.append({'modelo': nome, 'FP': int(fp), 'FN': int(fn), 'custo': int(CUSTO_FP * fp + CUSTO_FN * fn)})
print(CENARIO)
pd.DataFrame(custos).sort_values('custo')

In [ ]:
# A FAZER: o modelo escolhido pelo custo (o primeiro da tabela acima, ou outro com justificação)
ESCOLHA = pd.DataFrame(custos).sort_values('custo').modelo.iloc[0]
final = FAZ[ESCOLHA](BOTAO[ESCOLHA]).fit(X_tr, y_tr)
p_te = final.predict(X_te)
print('Escolhido:', ESCOLHA, '·', mede(ESCOLHA, y_te, p_te))

### O requisito comum: 1% de positivos

No vosso conjunto há muito mais positivos do que no mundo real. A função abaixo reamostra o **teste** para que só 1% sejam
positivos (mantém todos os negativos e sorteia positivos), vinte vezes, e dá a média. A segunda linha faz a mesma conta
sem sorteio, com a fórmula da aula 2 e da T05: precisão = rev × p ÷ (rev × p + TFP × (1 − p)).

In [ ]:
def com_prevalencia(y_real, y_prev, p=0.01, repeticoes=20):
    y_real, y_prev = np.asarray(y_real), np.asarray(y_prev)
    neg, pos = np.where(y_real == 0)[0], np.where(y_real == 1)[0]
    k = max(1, round(len(neg) * p / (1 - p)))
    precs, revs = [], []
    for s in range(repeticoes):
        idx = np.concatenate([neg, np.random.default_rng(s).choice(pos, size=min(k, len(pos)), replace=False)])
        precs.append(precision_score(y_real[idx], y_prev[idx], zero_division=0)); revs.append(recall_score(y_real[idx], y_prev[idx]))
    rev = recall_score(y_real, y_prev); tfp = ((y_prev == 1) & (y_real == 0)).sum() / len(neg)
    formula = rev * p / (rev * p + tfp * (1 - p)) if (rev * p + tfp * (1 - p)) > 0 else 0
    print(f'Com {100*p:.0f}% de positivos ({k} positivos e {len(neg)} negativos por amostra):')
    print(f'  reamostrado (média de {repeticoes}): precisão {np.mean(precs):.3f} · revocação {np.mean(revs):.3f}')
    print(f'  pela fórmula:                 precisão {formula:.3f} · revocação {rev:.3f}')

print('No teste tal como está:', mede(ESCOLHA, y_te, p_te))
com_prevalencia(y_te, p_te)

> **A FAZER (relatório, secção Escolha):** o cenário, a tabela de custos, o modelo escolhido e porquê, e o que acontece à
> precisão com 1% de positivos. Expliquem a queda com as vossas palavras: de cada 100 alertas, quantos são verdadeiros?

## Parte 5 · A pergunta obrigatória (passo 5)

**As características foram escolhidas em 1999 e algumas vêm da caixa de correio de uma única pessoa. Este detector serve para 2026? Como é que o testariam?**

Pista para começar: Olhem para as características que o vosso modelo mais usa (a função `mais_usadas` da Parte 5) e procurem o que cada uma é na documentação do conjunto no UCI.

A função abaixo mostra o que o modelo escolhido mais usa. É um ponto de partida, não a resposta.

In [ ]:
def mais_usadas(n=12):
    corr = X_tr.apply(lambda col: col.corr(y_tr)).abs()
    arv = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_tr, y_tr)
    imp = pd.Series(arv.feature_importances_, index=X_tr.columns)
    return pd.DataFrame({'|correlação com a etiqueta|': corr, 'importância numa árvore de profundidade 5': imp}).sort_values(
        'importância numa árvore de profundidade 5', ascending=False).head(n).round(3)

mais_usadas()

In [ ]:
# A FAZER: o vosso código para responder à pergunta obrigatória

> **A FAZER (relatório, secção Pergunta obrigatória):** a resposta, com os números que a sustentam.

## Parte 6 · Conclusão

> **A FAZER (relatório, secção Limites):** em três a cinco linhas, o que punham em produção, que número reportavam a um
> responsável de segurança, e o que este conjunto de dados **não** vos deixa saber.

---

**Entrega:** este caderno, corrido de ponta a ponta, e o relatório (máximo 6 páginas), no Moodle, até quinta-feira,
12 de Novembro, às 23h59. Se usaram IA para escrever código, digam onde no relatório.